In [ ]:
# ============================================================
# CELL 1: Git clone + Drive mount + Symlinks
# ============================================================

import os
import sys
import shutil
import subprocess
from pathlib import Path

def run_cmd(cmd, cwd=None, check=True):
    try:
        result = subprocess.run(
            cmd,
            cwd=cwd,
            shell=True,
            text=True,
            capture_output=True
        )
        if result.stdout:
            print(result.stdout.strip())
        if result.stderr:
            print(result.stderr.strip())
        if check and result.returncode != 0:
            raise RuntimeError(f"Command failed: {cmd}")
        return result
    except Exception as e:
        print(f"[ERROR] Failed to run command: {cmd}")
        print(f"[DETAIL] {e}")
        raise

print("=" * 70)
print("[INFO] Cell 1: Git clone + Drive mount + Symlinks")
print("=" * 70)

try:
    from google.colab import drive
    DRIVE_MOUNT = "/content/drive"
    REPO_URL = "https://github.com/tigerjs2/Fast-and-Robust-Crop-Disease-Prediction.git"
    BRANCH = "asusunha"
    REPO_DIR = "/content/Fast-and-Robust-Crop-Disease-Prediction"
    DRIVE_PROJECT_DIR = "/content/drive/MyDrive/Fast-and-Robust-Crop-Disease-Prediction"

    print("[INFO] Mounting Google Drive...")
    drive.mount(DRIVE_MOUNT, force_remount=True)
    print("[SUCCESS] Google Drive mounted.")

    if os.path.exists(REPO_DIR):
        print(f"[INFO] Removing existing repo directory: {REPO_DIR}")
        shutil.rmtree(REPO_DIR)

    print(f"[INFO] Cloning repository from branch '{BRANCH}'...")
    run_cmd(f"git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}")
    print("[SUCCESS] Repository cloned successfully.")

    os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)
    print(f"[INFO] Drive project directory ready: {DRIVE_PROJECT_DIR}")

    os.makedirs(f"{REPO_DIR}/drive_logs", exist_ok=True)
    os.makedirs(f"{REPO_DIR}/drive_artifacts", exist_ok=True)

    print(f"[SUCCESS] Cell 1 completed successfully.")
    print(f"[INFO] Repository directory: {REPO_DIR}")
    print(f"[INFO] Repository contents: {os.listdir(REPO_DIR)[:10]}")

except Exception as e:
    print("\n[FAILURE] Cell 1 failed.")
    print(f"[DETAIL] {e}")
    raise


In [ ]:
# ============================================================
# CELL 1.5: Drive에서 weights/data 폴더 복사
# ============================================================

import os
import shutil
from pathlib import Path

REPO_DIR = "/content/Fast-and-Robust-Crop-Disease-Prediction"
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/Fast-and-Robust-Crop-Disease-Prediction"

print("=" * 70)
print("[INFO] Cell 1.5: Drive에서 weights/data 폴더 복사")
print("=" * 70)

try:
    folders_to_copy = ["weights", "data"]

    for folder_name in folders_to_copy:
        src_path = os.path.join(DRIVE_PROJECT_DIR, folder_name)
        dst_path = os.path.join(REPO_DIR, folder_name)

        if os.path.exists(src_path):
            print(f"[INFO] 복사 중: {folder_name}")
            if os.path.exists(dst_path):
                shutil.rmtree(dst_path)
            shutil.copytree(src_path, dst_path)
            print(f"[SUCCESS] {folder_name} 복사 완료")
            print(f"  src: {src_path}")
            print(f"  dst: {dst_path}")
        else:
            print(f"[WARNING] Drive에서 찾을 수 없음: {src_path}")
            print(f"[ACTION] Drive에 {folder_name} 폴더를 다음 경로에 만들고 파일 추가:")
            print(f"  {DRIVE_PROJECT_DIR}/{folder_name}")

    print("\n[INFO] 복사 후 repo 디렉토리 확인:")
    contents = os.listdir(REPO_DIR)
    for item in sorted(contents):
        item_path = os.path.join(REPO_DIR, item)
        if os.path.isdir(item_path):
            size = sum(f.stat().st_size for f in Path(item_path).rglob('*') if f.is_file())
            print(f"  📁 {item}/ ({size / (1024**3):.2f} GB)")
        else:
            print(f"  📄 {item}")

    print("\n[SUCCESS] Cell 1.5 completed.")

except Exception as e:
    print("\n[FAILURE] Cell 1.5 failed.")
    print(f"[DETAIL] {e}")
    raise


In [ ]:
# ============================================================
# CELL 2: pip install requirements.txt (server/ 경로)
# ============================================================

import os
import subprocess
import sys
from pathlib import Path

REPO_DIR = "/content/Fast-and-Robust-Crop-Disease-Prediction"
REQ_FILE = os.path.join(REPO_DIR, "server", "requirements.txt")  # ← server/ 추가
INSTALL_LOG = os.path.join(REPO_DIR, "drive_logs", "pip_install.log")

def run_cmd(cmd, cwd=None, check=True, log_file=None):
    try:
        result = subprocess.run(
            cmd,
            cwd=cwd,
            shell=True,
            text=True,
            capture_output=True
        )

        if log_file:
            os.makedirs(os.path.dirname(log_file), exist_ok=True)
            with open(log_file, "a", encoding="utf-8") as f:
                f.write(f"\n\n$ {cmd}\n")
                f.write(result.stdout or "")
                f.write("\n")
                f.write(result.stderr or "")
                f.write("\n")

        if result.stdout:
            print(result.stdout.strip()[:4000])
        if result.stderr:
            print(result.stderr.strip()[:4000])

        if check and result.returncode != 0:
            raise RuntimeError(f"Command failed: {cmd}")
        return result
    except Exception as e:
        print(f"[ERROR] Failed command: {cmd}")
        print(f"[DETAIL] {e}")
        raise

print("=" * 70)
print("[INFO] Cell 2: pip install requirements.txt")
print("=" * 70)

try:
    if not os.path.exists(REQ_FILE):
        raise FileNotFoundError(f"requirements.txt not found at: {REQ_FILE}")

    print(f"[INFO] requirements.txt found: {REQ_FILE}")
    print("[INFO] Upgrading pip/setuptools/wheel...")
    run_cmd("python -m pip install --upgrade pip setuptools wheel", log_file=INSTALL_LOG)

    print("[INFO] Installing dependencies from requirements.txt...")
    run_cmd(f"pip install -r '{REQ_FILE}'", cwd=REPO_DIR, log_file=INSTALL_LOG)

    print("[INFO] Installing helper packages (pyngrok, requests, pillow)...")
    run_cmd("pip install pyngrok requests pillow", log_file=INSTALL_LOG)

    print(f"\n[SUCCESS] Dependencies installed successfully.")
    print(f"[INFO] Install log saved to: {INSTALL_LOG}")

except Exception as e:
    print("\n[FAILURE] Cell 2 failed.")
    print(f"[DETAIL] {e}")
    if os.path.exists(INSTALL_LOG):
        print(f"[INFO] Check install log here: {INSTALL_LOG}")
    raise


In [ ]:
# ============================================================
# CELL 3: PYTHONPATH setup + Uvicorn startup
# ============================================================

import os
import sys
import time
import signal
import subprocess
from pathlib import Path

REPO_DIR = "/content/Fast-and-Robust-Crop-Disease-Prediction"
SERVER_DIR = os.path.join(REPO_DIR, "server")  # ← server/ 추가
SERVER_LOG = os.path.join(REPO_DIR, "drive_logs", "uvicorn.log")
SERVER_PID_FILE = os.path.join(REPO_DIR, "drive_logs", "uvicorn.pid")
PORT = 8000

APP_MODULE_CANDIDATES = [
    "main:app",
    "app.main:app",
    "src.main:app",
]

print("=" * 70)
print("[INFO] Cell 3: PYTHONPATH setup + Uvicorn startup")
print("=" * 70)

def file_tail(path, n=40):
    if not os.path.exists(path):
        return "[INFO] Log file does not exist yet."
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()
    return "".join(lines[-n:])

try:
    if not os.path.exists(SERVER_DIR):
        raise FileNotFoundError(f"Server directory not found: {SERVER_DIR}")

    os.environ["PYTHONPATH"] = SERVER_DIR + (":" + os.environ.get("PYTHONPATH", ""))
    print(f"[SUCCESS] PYTHONPATH set to include: {SERVER_DIR}")

    if os.path.exists(SERVER_PID_FILE):
        try:
            with open(SERVER_PID_FILE, "r") as f:
                old_pid = int(f.read().strip())
            os.kill(old_pid, signal.SIGTERM)
            print(f"[INFO] Stopped previous server process (PID: {old_pid})")
            time.sleep(2)
        except Exception as e:
            print(f"[WARNING] Could not stop previous process: {e}")

    selected_module = None
    launch_errors = []

    os.makedirs(os.path.dirname(SERVER_LOG), exist_ok=True)

    for app_module in APP_MODULE_CANDIDATES:
        print(f"[INFO] Trying app module: {app_module}")

        with open(SERVER_LOG, "w", encoding="utf-8") as logf:
            process = subprocess.Popen(
                [
                    sys.executable, "-m", "uvicorn",
                    app_module,
                    "--host", "0.0.0.0",
                    "--port", str(PORT)
                ],
                cwd=SERVER_DIR,  # ← SERVER_DIR로 변경
                stdout=logf,
                stderr=logf,
                env=os.environ.copy()
            )

        time.sleep(8)

        if process.poll() is None:
            selected_module = app_module
            with open(SERVER_PID_FILE, "w") as f:
                f.write(str(process.pid))
            print(f"[SUCCESS] Uvicorn started with {app_module}")
            print(f"[INFO] PID: {process.pid}")
            break
        else:
            err_log = file_tail(SERVER_LOG, 60)
            launch_errors.append((app_module, err_log))
            print(f"[WARNING] Failed: {app_module}")

    if selected_module is None:
        raise RuntimeError("All app modules failed.")

    print(f"[INFO] Server log: {SERVER_LOG}")
    print("[INFO] Recent logs:")
    print(file_tail(SERVER_LOG, 40))
    print("\n[SUCCESS] Cell 3 completed.")

except Exception as e:
    print("\n[FAILURE] Cell 3 failed.")
    print(f"[DETAIL] {e}")
    raise


In [ ]:
# ============================================================
# CELL 4: ngrok tunnel connection
# ============================================================

import os
import time
from pyngrok import ngrok, conf

PORT = 8000
NGROK_TOKEN = "3EJG6H6Oq2iU9Xc0Y4b5jm4D8BI_2wHGZKcPkLFgbCggQFZVL"
REPO_DIR = "/content/Fast-and-Robust-Crop-Disease-Prediction"
NGROK_URL_FILE = os.path.join(REPO_DIR, "drive_logs", "ngrok_url.txt")

print("=" * 70)
print("[INFO] Cell 4: ngrok tunnel connection")
print("=" * 70)

try:
    print("[INFO] Setting ngrok auth token...")
    ngrok.set_auth_token(NGROK_TOKEN)
    print("[SUCCESS] ngrok auth token configured.")

    print("[INFO] Cleaning up existing ngrok tunnels...")
    try:
        ngrok.kill()
        time.sleep(2)
    except Exception as e:
        print(f"[WARNING] ngrok cleanup: {e}")

    print(f"[INFO] Opening ngrok tunnel to port {PORT}...")
    public_tunnel = ngrok.connect(PORT, "http")
    public_url = public_tunnel.public_url

    os.makedirs(os.path.dirname(NGROK_URL_FILE), exist_ok=True)
    with open(NGROK_URL_FILE, "w", encoding="utf-8") as f:
        f.write(public_url)

    print(f"[SUCCESS] ngrok tunnel established.")
    print(f"\n[PUBLIC URL] {public_url}")
    print(f"[INFO] Saved to: {NGROK_URL_FILE}")
    print("\n[SUCCESS] Cell 4 completed.")

except Exception as e:
    print("\n[FAILURE] Cell 4 failed.")
    print(f"[DETAIL] {e}")
    raise


In [ ]:
# ============================================================
# CELL 5: Server health check
# ============================================================

import os
import time
import requests

REPO_DIR = "/content/Fast-and-Robust-Crop-Disease-Prediction"
NGROK_URL_FILE = os.path.join(REPO_DIR, "drive_logs", "ngrok_url.txt")
SERVER_LOG = os.path.join(REPO_DIR, "drive_logs", "uvicorn.log")

print("=" * 70)
print("[INFO] Cell 5: Server health check")
print("=" * 70)

def tail_log(path, n=50):
    if not os.path.exists(path):
        return "[INFO] No log file."
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()
    return "".join(lines[-n:])

try:
    if not os.path.exists(NGROK_URL_FILE):
        raise FileNotFoundError("ngrok URL file not found. Run Cell 4 first.")

    with open(NGROK_URL_FILE, "r", encoding="utf-8") as f:
        base_url = f.read().strip()

    print(f"[INFO] Using base URL: {base_url}")

    endpoints = ["/health", "/crops", "/classes"]
    results = {}

    for ep in endpoints:
        url = base_url + ep
        print(f"\n[INFO] Checking: {url}")
        try:
            response = requests.get(url, timeout=30)
            results[ep] = {
                "status_code": response.status_code,
                "body": response.text[:500]
            }
            if response.ok:
                print(f"[SUCCESS] {ep} HTTP {response.status_code}")
                print(f"[BODY] {response.text[:300]}")
            else:
                print(f"[WARNING] {ep} HTTP {response.status_code}")
        except Exception as e:
            print(f"[FAILURE] {ep} failed: {e}")

    print("\n[INFO] Server logs:")
    print(tail_log(SERVER_LOG, 60))
    print("\n[SUCCESS] Cell 5 completed.")

except Exception as e:
    print("\n[FAILURE] Cell 5 failed.")
    print(f"[DETAIL] {e}")
    raise


In [ ]:
# ============================================================
# CELL 6: /predict test with sample image
# ============================================================

import os
import requests
from PIL import Image, ImageDraw
import numpy as np

REPO_DIR = "/content/Fast-and-Robust-Crop-Disease-Prediction"
NGROK_URL_FILE = os.path.join(REPO_DIR, "drive_logs", "ngrok_url.txt")
SERVER_LOG = os.path.join(REPO_DIR, "drive_logs", "uvicorn.log")
SAMPLE_IMAGE = os.path.join(REPO_DIR, "drive_artifacts", "sample_leaf.jpg")

print("=" * 70)
print("[INFO] Cell 6: /predict test with sample image")
print("=" * 70)

def tail_log(path, n=60):
    if not os.path.exists(path):
        return "[INFO] No log file."
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()
    return "".join(lines[-n:])

try:
    if not os.path.exists(NGROK_URL_FILE):
        raise FileNotFoundError("ngrok URL file not found. Run Cell 4 first.")

    with open(NGROK_URL_FILE, "r", encoding="utf-8") as f:
        base_url = f.read().strip()

    predict_url = base_url + "/predict"
    print(f"[INFO] Predict URL: {predict_url}")

    if not os.path.exists(SAMPLE_IMAGE):
        print("[INFO] Creating sample test image...")
        os.makedirs(os.path.dirname(SAMPLE_IMAGE), exist_ok=True)
        img = Image.new("RGB", (256, 256), color=(34, 139, 34))
        draw = ImageDraw.Draw(img)
        draw.ellipse((60, 60, 200, 200), fill=(50, 205, 50), outline=(0, 100, 0), width=5)
        draw.line((80, 180, 180, 80), fill=(139, 69, 19), width=6)
        img.save(SAMPLE_IMAGE)
        print(f"[SUCCESS] Sample image created: {SAMPLE_IMAGE}")
    else:
        print(f"[INFO] Using existing sample image: {SAMPLE_IMAGE}")

    with open(SAMPLE_IMAGE, "rb") as f:
        files = {
            "image": ("sample_leaf.jpg", open(SAMPLE_IMAGE, "rb"), "image/jpeg"),
        }
        data = {
            "crop": "tomato",
            "bbox": "0,0,255,255",  # 256x256 이미지 기준
            "return_masked_image": "false",
        }
        response = requests.post(predict_url, files=files, data=data, timeout=60)

    print(f"[INFO] HTTP status: {response.status_code}")
    print("[INFO] Response preview:")
    print(response.text[:1500])

    if response.ok:
        print("\n[SUCCESS] /predict test completed.")
    else:
        print("\n[WARNING] /predict returned non-2xx.")
        print("[DEBUG] Server logs:")
        print(tail_log(SERVER_LOG, 80))

    print("\n[SUCCESS] Cell 6 completed.")

except Exception as e:
    print("\n[FAILURE] Cell 6 failed.")
    print(f"[DETAIL] {e}")
    raise


In [ ]:
# Colab 셀에서 다음 코드 실행
import os

REPO_DIR = "/content/Fast-and-Robust-Crop-Disease-Prediction"
NGROK_URL_FILE = os.path.join(REPO_DIR, "drive_logs", "ngrok_url.txt")

with open(NGROK_URL_FILE, "r", encoding="utf-8") as f:
    public_url = f.read().strip()

print(f"🔗 공개 URL: {public_url}")
